#### **Supplementary Code 1**
---
### Targetted Enzyme Discovery using Metal-Coordination Mining
####         Ioannis Kipouros &  Michelle Chang (Nature, 2026)

In [ ]:
# import all libraries here:
import os, sys, errno, re, json, ssl
from pathlib import Path
from urllib import request
from urllib.error import HTTPError
import time 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from shutil import copyfile
from Bio.PDB import *
import datetime as dt
from io import StringIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep


### 0. Project Structure and Required Inputs

Set the `PROJECT_ROOT` directory in the code cell below.

Within this directory:

- Place all required input files in the `inputs/` folder.
- All results will be written automatically to `outputs/<RUN_ID>/` (a new timestamped folder is created for each run).
- No other file paths need to be edited.

---

### Required Input Files

**After halogenases are identified (external steps required):**
- `idmapping.fasta`  
  (Downloaded from UniProt ID mapping tool using putative halogenase accessions)

- `accession_IDs_in_SSN.csv`  
  (Generated from EFI-EST SSN analysis; must contain columns: `all_hals`, `known_hals`, `new_hals`)

- `Table_S1_proteinFamilies.csv`  
  (Template table for generating *Supplementary Table 1*)

All other files are generated automatically during execution.


In [ ]:
# ===== User sets ONE path =====
# All inputs will be read from:  <PROJECT_ROOT>/inputs
# All outputs will be written to: <PROJECT_ROOT>/outputs/<RUN_ID>/
PROJECT_ROOT = Path(
    #"pathfile_here"  # <-- user edits only this line
).expanduser().resolve()

# ===== Input folder =====
INPUT_DIR = PROJECT_ROOT / "inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Output folder =====
RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTDIR = PROJECT_ROOT / "outputs" / RUN_ID

# ===== Define the standard structure for the code output =====
DIRS = {
    "0_new_IDs": OUTDIR / "0_new_IDs",
    "1_protein_families": OUTDIR / "1_protein_families",
    "2_retrieved_accessionIDs": OUTDIR / "2_retrieved_accessionIDs",
    "3_processed_accessionIDs": OUTDIR / "3_processed_accessionIDs",
    "4_retrieved_AF2modelsche": OUTDIR / "4_retrieved_AF2modelsche",
    "5_find_2His": OUTDIR / "5_find_2His",
    "6_retrieved_halgogenases_from2HisSite": OUTDIR / "6_retrieved_halgogenases_from2HisSite",
    "7_getUNIPROTannotations_IDmapping": OUTDIR / "7_getUNIPROTannotations_IDmapping",
    "8_TableS1": OUTDIR / "8_TableS1",
}

# Create everything
for p in [OUTDIR, *DIRS.values()]:
    p.mkdir(parents=True, exist_ok=True)

print("Input folder:", INPUT_DIR)
print("Writing outputs to:", OUTDIR)


### 1. Define the Protein Sequence Space for Structural Mining

- The PFAM “Cupin clan” (`CL0029`), which comprises 80 (super)families containing the β-barrel cupin fold—the characteristic structural scaffold of Fe(II)/αKG-dependent enzymes was used as the starting point to define the initial protein sequence space. The InterPro entry `IPR042098` ("Taurine dioxygenase TauD-like superfamily") was additionally included. This is represented below as the dataframe named `df_CL0029`

- The presence of a cupin domain was used as the primary inclusion criterion to generate a high-coverage sequence dataset for subsequent structural mining of radical halogenases.

- The resulting protein groups were manually curated to remove families corresponding to sugar epimerases, transcription factors, and seed storage proteins, leading to inclusion of 48 out of the 81 superfamilies (listed in *Supplementary Table 1* and identified with a *T* boolean value in the *included* column of `df_CL0029` ). This filtering step reduced dataset size and lowered the expected false-positive rate, as these excluded groups are not anticipated to bind α-ketoglutarate (aKG). This assumes broad functional conservation within sequence-based families (i.e., aKG-dependent vs. aKG-independent cupins), and our analysis specifically focuses on identifying radical halogenases within aKG-dependent enzymes.


In [ ]:
 
df_CL0029 = """Accession\tName\tSource Database\tType\tIntegrated Into\tinclude
PF00027\tCyclic nucleotide-binding domain\tpfam\tdomain\tIPR000595\tF
PF00190\tCupin\tpfam\tdomain\tIPR006045\tF
PF00908\tdTDP-4-dehydrorhamnose 3,5-epimerase\tpfam\tdomain\tIPR000888\tF
PF01050\tMannose-6-phosphate isomerase C-terminal\tpfam\tfamily\tIPR001538\tF
PF01238\tPhosphomannose isomerase type I C-terminal\tpfam\tdomain\tIPR046456\tF
PF01987\tMitochondrial biogenesis AIM24\tpfam\tfamily\tIPR002838\tF
PF02041\tAuxin binding protein\tpfam\tdomain\tIPR000526\tF
PF02311\tAraC-like ligand binding domain\tpfam\tdomain\tIPR003313\tF
PF02373\tJmjC domain, hydroxylase\tpfam\tdomain\tIPR003347\tT
PF02375\tjmjN domain\tpfam\tfamily\tIPR003349\tT
PF02668\tTaurine catabolism dioxygenase TauD, TfdA family\tpfam\tdomain\tIPR003819\tT
PF02678\tPirin\tpfam\tfamily\tIPR003829\tF
PF03079\tARD/ARD' family\tpfam\tdomain\tIPR004313\tT
PF03171\t2OG-Fe(II) oxygenase superfamily\tpfam\tdomain\tIPR044861\tT
PF03336\tPoxvirus C4/C10 protein\tpfam\tfamily\tIPR005004\tT
PF04074\tYhcH/YjgK/YiaL\tpfam\tdomain\tIPR004375\tT
PF04115\tUreidoglycolate lyase\tpfam\tdomain\tIPR007247\tT
PF04209\tHomogentisate 1,2-dioxygenase C-terminal\tpfam\tdomain\tIPR046451\tT
PF04622\tERG2 and Sigma1 receptor like protein\tpfam\tfamily\tIPR006716\tF
PF04831\tPOPDC1-3\tpfam\tdomain\tIPR055272\tF
PF04962\tKduI/IolB family\tpfam\tdomain\tIPR021120\tF
PF05118\tAspartyl/Asparaginyl beta-hydroxylase\tpfam\tdomain\tIPR007803\tT
PF05523\tWxcM-like, C-terminal\tpfam\tdomain\tIPR008894\tF
PF05721\tPhytanoyl-CoA dioxygenase (PhyH)\tpfam\tdomain\tIPR008775\tT
PF05726\tPirin C-terminal cupin domain\tpfam\tdomain\tIPR008778\tF
PF05899\tEutQ-like cupin domain\tpfam\tdomain\tIPR008579\tF
PF05962\tHutD\tpfam\tfamily\tIPR010282\tF
PF05995\tCysteine dioxygenase type I\tpfam\tdomain\tIPR010300\tT
PF06052\t3-hydroxyanthranilic acid dioxygenase\tpfam\tdomain\tIPR010329\tT
PF06172\tCupin superfamily (DUF985)\tpfam\tdomain\tIPR009327\tT
PF06249\tEthanolamine utilisation protein EutQ\tpfam\tdomain\tIPR010424\tF
PF06339\tEctoine synthase\tpfam\tfamily\tIPR010462\tT
PF06560\tGlucose-6-phosphate isomerase (GPI)\tpfam\tdomain\tIPR010551\tF
PF06719\tAraC-type transcriptional regulator N-terminus\tpfam\tfamily\tIPR009594\tF
PF06865\tPyrimidine/purine nucleoside phosphorylase\tpfam\tdomain\tIPR009664\tF
PF07350\tGig2-like\tpfam\tfamily\tIPR010856\tT
PF07385\tD-lyxose isomerise\tpfam\tdomain\tIPR010864\tF
PF07847\tPCO_ADO\tpfam\tfamily\tIPR012864\tT
PF07883\tCupin domain\tpfam\tdomain\tIPR013096\tT
PF08007\tJmjC domain\tpfam\tdomain\tIPR003347\tT
PF08487\tVault protein inter-alpha-trypsin domain\tpfam\tfamily\tIPR013694\tF
PF08943\tCsiD\tpfam\tdomain\tIPR015038\tT
PF09313\tDomain of unknown function (DUF1971)\tpfam\tdomain\tIPR015392\tF
PF09859\tOxygenase, catalysing oxidative methylation of damaged DNA\tpfam\tdomain\tIPR018655\tT
PF10014\t2OG-Fe dioxygenase\tpfam\tdomain\tIPR018724\tT
PF10637\tOxoglutarate and iron-dependent oxygenase degradation C-term\tpfam\tdomain\tIPR019601\tT
PF11004\t3-deoxy-D-manno-oct-2-ulosonic acid (Kdo) hydroxylase\tpfam\tfamily\tIPR021266\tT
PF11699\tMif2/CENP-C like\tpfam\tdomain\tIPR025974\tF
PF12851\tOxygenase domain of the 2OGFeDO superfamily\tpfam\tdomain\tIPR024779\tT
PF12852\tCupin\tpfam\tdomain\tIPR032783\tT
PF12933\tFTO catalytic domain\tpfam\tdomain\tIPR024367\tT
PF12973\tChrR Cupin-like domain\tpfam\tdomain\tIPR025979\tF
PF13532\t2OG-Fe(II) oxygenase superfamily\tpfam\tdomain\tIPR027450\tT
PF13621\tCupin-like domain\tpfam\tdomain\tIPR041667\tT
PF13640\t2OG-Fe(II) oxygenase superfamily\tpfam\tdomain\tIPR044862\tT
PF13661\t2OG-Fe(II) oxygenase superfamily\tpfam\tdomain\tIPR039558\tT
PF13757\tVault protein inter-alpha-trypsin domain\tpfam\tdomain\tIPR013694\tF
PF13759\tPutative 2OG-Fe(II) oxygenase\tpfam\tdomain\tIPR012668\tT
PF14033\tProtein of unknown function (DUF4246)\tpfam\tdomain\tIPR049192\tT
PF14226\tnon-haem dioxygenase in morphine synthesis N-terminal\tpfam\tfamily\tIPR026992\tT
PF14499\tDomain of unknown function (DUF4437)\tpfam\tfamily\tIPR028013\tT
PF14525\tAraC-binding-like domain\tpfam\tdomain\tIPR035418\tF
PF16161\tDomain of unknown function (DUF4867)\tpfam\tfamily\tIPR032358\tT
PF16802\tDomain of unknown function (DUF5070)\tpfam\tfamily\tIPR031839\tT
PF16867\tDimethlysulfonioproprionate lyase\tpfam\tdomain\tIPR031723\tT
PF17954\tQuercetinase C-terminal cupin domain\tpfam\tdomain\tIPR041602\tT
PF18637\tAldos-2-ulose dehydratase/isomerase (AUDH) Cupin domain\tpfam\tdomain\tIPR040887\tF
PF19480\tDomain of unknown function (DUF6016)\tpfam\tdomain\tIPR046058\tT
PF20510\tHomogentisate 1,2-dioxygenase N-terminal\tpfam\tdomain\tIPR046452\tT
PF20511\tPhosphomannose isomerase type I, catalytic domain\tpfam\tdomain\tIPR046457\tF
PF20515\tTet-like 2OG-Fe(II) oxygenase superfamily\tpfam\tdomain\tIPR046798\tT
PF21621\tMannose-6-phosphate isomerase, cupin domain\tpfam\tdomain\tIPR049071\tF
PF22200\tExsA N-terminal regulatory domain\tpfam\tdomain\tIPR054015\tF
PF22404\tToxT, N-terminal cupin-like domain\tpfam\tdomain\tIPR055025\tF
PF22814\tCarrier-protein-independent halogenase WelO5\tpfam\tfamily\tIPR055091\tT
PF23169\tHalogenase D\tpfam\tdomain\tIPR056470\tT
PF24419\tPolynucleotide 5'-hydroxyl-kinase NOL9, N-terminal domain\tpfam\tdomain\tNA\tF
SSF51197\tClavaminate synthase-like\tINTERPRO\tsuperfamily\tNA\tT
IPR005123\tOxoglutarate/iron-dependent dioxygenase domain\tINTERPRO\tdomain\tIPR005123\tT
IPR056470\tL-lysine 4-chlorinase BesD/L-lysine 5-chlorinase HalB-like\tINTERPRO\tfamily\tNA\tT
IPR042098\tTaurine dioxygenase TauD-like superfamily\tINTERPRO\tsuperfamily\tNA\tT
"""

df_all_families = pd.read_csv(StringIO(df_CL0029), sep="\t")
print(df_all_families.head(10))

In [ ]:
### === RETRIEVE ACCESSION IDs FROM INTERPRO ===

# Output folder for this step
OUT_ACCESSIONS_DIR = DIRS["2_retrieved_accessionIDs"]
OUT_ACCESSIONS_DIR.mkdir(parents=True, exist_ok=True)

# Input: a family ID (PFAM / INTERPRO / SSF)
# Output: a table written to "<ID>_accessions_retrieved.csv"
#         columns: AccessionID,Name,Length,Gene,in_alphafold

def retrieve_accessionIDs_from_interpro(input_id, out_dir=OUT_ACCESSIONS_DIR):
    input_id = str(input_id).strip()

    if input_id.startswith("I"):
        base_url = f"https://www.ebi.ac.uk/interpro/api/protein/UniProt/entry/InterPro/{input_id}/?page_size=200"
    elif input_id.startswith("P"):
        base_url = f"https://www.ebi.ac.uk/interpro/api/protein/UniProt/entry/PFAM/{input_id}/?page_size=200"
    elif input_id.startswith("S"):
        base_url = f"https://www.ebi.ac.uk/interpro/api/protein/UniProt/entry/ssf/{input_id}/?page_size=200"
    else:
        raise ValueError(f"Unsupported input ID: {input_id}")

    context = ssl._create_unverified_context()

    out_path = out_dir / f"{input_id}_accessions_retrieved.csv"
    out_path.write_text("AccessionID,Name,Length,Gene,in_alphafold\n")

    next_page = base_url
    total_results = 0
    next_report = 10000  # threshold for next progress print

    print(f"Currently downloading family {input_id}...")

    while next_page:
        try:
            req = request.Request(next_page, headers={"Accept": "application/json"})
            res = request.urlopen(req, context=context)

            status = res.getcode()
            if status == 408:
                sleep(61)
                continue
            elif status == 204:
                break

            payload = json.loads(res.read().decode())
            next_page = payload.get("next")

            results = payload.get("results") or []
            if not results:
                break

            with out_path.open("a") as f:
                for item in results:
                    md = item.get("metadata", {})
                    f.write(
                        f"{md.get('accession', 'N/A')},"
                        f"{md.get('name', 'N/A')},"
                        f"{md.get('length', 'N/A')},"
                        f"{md.get('gene', 'N/A')},"
                        f"{md.get('in_alphafold', 'N/A')}\n"
                    )

            total_results += len(results)

            if total_results >= next_report:
                print(f"{total_results:,} accessions retrieved for {input_id}")
                next_report += 10000

            sleep(1)  # be polite to the server

        except (HTTPError, json.JSONDecodeError) as e:
            sys.stderr.write(f"[{input_id}] Error: {e}\n")
            sleep(5)

    print(f"Family {input_id} downloaded, {total_results:,} total accessions retrieved")
    return out_path


# Retrieve accession IDs for each family marked for inclusion
for fam_id in df_all_families.loc[df_all_families["include"] == "T", "Accession"]:
    out_file = retrieve_accessionIDs_from_interpro(fam_id)
    print(f"Wrote: {out_file}")

#### Compile and Validate Retrieved Accession IDs

This step aggregates all accession tables retrieved from InterPro into a single curated dataset.

Specifically, the script:
- Reads all `.txt` files from the `2_retrieved_accessionIDs` output directory.
- Validates UniProt accession IDs using a regular expression filter.
- Annotates each entry with its corresponding protein family ID.
- Compiles all families into a single dataframe.
- Generates summary counts of unique accessions per family.

The compiled dataset and family-level summary tables are written to the `3_processed_accessionIDs` output directory for downstream analysis.


In [ ]:

# === Settings ===
IN_ACCESSIONS_DIR = DIRS["2_retrieved_accessionIDs"]          
OUT_PROCESSED_DIR = DIRS["3_processed_accessionIDs"]        
OUT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Expected column names in the retrieved .csv files
column_names = ["AccessionID", "Name", "Length", "Gene", "in_alphafold"]

# UniProt accession regex (6-char or 10-char, official-style formats)
# uniprot_regex = re.compile(r'^[A-Z0-9]{6,10}$')

def is_valid_uniprot(s) -> bool:
    """Return True if s looks like a valid UniProt accession."""
    if s is None:
        return False
    s = str(s).strip()
    return bool(uniprot_regex.match(s))


# ==== 1. Compile into one df, validate accessions, add family annotation ===
dataframes = []

csv_files = sorted(IN_ACCESSIONS_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(
        f"No .csv files found in: {IN_ACCESSIONS_DIR}\n"
        "Did you run the InterPro accession retrieval step?"
    )

for file_path in csv_files:
    filename = file_path.name

    # Peek at the first line to see if it's a header
    with file_path.open("r") as f:
        first_line = f.readline().strip().split(",")

    if first_line == column_names:
        # File has a proper header row
        df = pd.read_csv(
            file_path,
            sep=",",
            skip_blank_lines=True,
            header=0,
            on_bad_lines="skip",
            na_filter=True
        )
    else:
        # No header → treat all rows as data, assign our own names
        df = pd.read_csv(
            file_path,
            sep=",",
            skip_blank_lines=True,
            names=column_names,
            header=None,
            on_bad_lines="skip",
            na_filter=True
        )

    # Keep only expected columns (in case files have extras)
    df = df[[c for c in column_names if c in df.columns]].copy()

    # Keep only rows with a valid UniProt accession
    #df = df[df["AccessionID"].apply(is_valid_uniprot)].copy()

    # Add family ID from filename (prefix before first "_")
    # e.g., "IPRxxxxx_accessions_retrieved.csv" -> "IPRxxxxx"
    family_id = filename.split("_")[0]
    df["family_ID"] = family_id

    dataframes.append(df)

# Combine all families into one dataframe
df_all_retrieved = pd.concat(dataframes, ignore_index=True) if dataframes else pd.DataFrame(columns=column_names + ["family_ID"])

# Save compiled dataframe with timestamp
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
compiled_path = OUT_PROCESSED_DIR / f"compiled_accession_IDs_{timestamp}.csv"
df_all_retrieved.to_csv(compiled_path, index=False)

print(f"Compiled dataframe saved to: {compiled_path}")
print(f"Total rows after UniProt filtering: {df_all_retrieved.shape[0]}")


# === 2. Count number of unique accessions per family ===
family_counts = (
    df_all_retrieved
    .groupby("family_ID")["AccessionID"]
    .nunique()
    .reset_index()
    .rename(columns={"AccessionID": "accession_count"})
)

counts_path = OUT_PROCESSED_DIR / f"accession_counts_by_family_{timestamp}.csv"
family_counts.to_csv(counts_path, index=False)

print(f"Family accession counts saved to: {counts_path}")


Some UniProt accessions are associated with multiple InterPro/PFAM families.  
To avoid redundancy in downstream structural analysis, accession IDs are collapsed to unique entries.

Specifically:
- Rows with missing accession IDs are removed.
- Entries are grouped by `AccessionID`.
- Metadata fields (`Name`, `Length`, `Gene`, `in_alphafold`) retain the first occurrence.
- Family annotations are merged into a single string ("/"-separated) if an accession appears in multiple families.

The resulting non-redundant dataset is written to the `3_processed_accessionIDs` output directory.


In [ ]:
# Remove rows without valid accession IDs
df_all_retrieved = df_all_retrieved[
    df_all_retrieved["AccessionID"].notna() &
    (df_all_retrieved["AccessionID"].astype(str).str.strip() != "")
]

# Collapse duplicate accessions across families
df_all_retrieved_unique = (
    df_all_retrieved
    .groupby("AccessionID", sort=False)
    .agg({
        "Name": "first",
        "Length": "first",
        "Gene": "first",
        "in_alphafold": "first",
        "family_ID": lambda x: "/".join(sorted(set(x.dropna())))
    })
    .reset_index()
)

print(df_all_retrieved_unique.head(10))
print(f"Unique accessions: {df_all_retrieved_unique.shape[0]}")

# Save unique accession dataset
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = DIRS["3_processed_accessionIDs"] / \
    f"accessions_withAF2_morethan150aa_noduplicates_{timestamp}.csv"

df_all_retrieved_unique.to_csv(output_file, index=False)

print(f"Saved non-redundant accession dataset to: {output_file}")


The non-redundant accession dataset is further filtered to define the final protein set used for structural mining.

Specifically:
- Only sequences with available AlphaFold2 models are retained.
- A minimum protein length cutoff (>150 amino acids) is applied to exclude truncated or partial sequences.

The resulting dataset represents the curated set of candidate cupin-domain proteins used for downstream AF2 model retrieval and structural analysis.


In [ ]:
# Convert Length column to numeric
df_all_retrieved_unique["Length"] = pd.to_numeric(
    df_all_retrieved_unique["Length"], errors="coerce"
)

# Normalize in_alphafold to a boolean
df_all_retrieved_unique["in_alphafold"] = (
    df_all_retrieved_unique["in_alphafold"]
    .astype(str).str.strip().str.lower()
    .map({"true": True, "false": False})
    .fillna(False)
)

# Apply filtering criteria
df_retrieved_clean = df_all_retrieved_unique[
    (df_all_retrieved_unique["in_alphafold"]) &
    (df_all_retrieved_unique["Length"] > 150)
].copy()

# Save filtered dataset for AF2 retrieval step
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
filtered_output = DIRS["3_processed_accessionIDs"] / f"accessions_AF2_lengthGT150_{timestamp}.csv"
df_retrieved_clean.to_csv(filtered_output, index=False)

# Summary statistics
total_retrieved = df_all_retrieved.shape[0]
total_unique = df_all_retrieved_unique.shape[0]
with_af2 = df_all_retrieved_unique["in_alphafold"].sum()  # counts True values
final_count = df_retrieved_clean.shape[0]

print(f"Total retrieved sequences: {total_retrieved}")
print(f"Unique accession IDs after duplicate removal: {total_unique}")
print(f"Accessions with AF2 models: {with_af2}")
print(f"Accessions with AF2 models and length >150 aa: {final_count}")
print(f"Filtered dataset saved to: {filtered_output}")


## Part 2. Retrieval of AlphaFold2 Structural Models

AlphaFold2 models corresponding to the filtered accession list (Part 1) are retrieved from the AlphaFold Protein Structure Database (AFDB).

For each UniProt accession:
- The AFDB prediction API is queried to obtain the appropriate PDB download URL for the corresponding model.
- The PDB file is downloaded and saved to the `4_retrieved_AF2modelsche` output directory.
- A log file is generated listing successfully retrieved models, skipped entries (already present), and failed retrievals.

This step defines the structural dataset used for subsequent active-site mining.


In [ ]:
# === RETRIEVE ALPHAFOLD2 MODELS FROM AF2DB API ===
#
#   1. Loads the most recent filtered accession list generated
#      in the previous step of the pipeline.
#   2. Queries the AlphaFold2 (AF2) Database (AFDB) API for each accession
#      to obtain the corresponding PDB download URL.
#   3. Downloads AF2 models in parallel to improve speed.
#   4. Prints progress messages during the run, including:
#         - how many structures have been successfully downloaded
#           (reported every 10,000 downloads)
#         - occasional overall processing updates
#   5. Writes a timestamped download log at the end.
#
# Notes:
#   - Existing non-empty PDB files are skipped.
#   - Failed downloads are recorded in the log.


# === Inputs / Outputs ===
PROCESSED_DIR = DIRS["3_processed_accessionIDs"]
AF2_OUTPUT_DIR = DIRS["4_retrieved_AF2modelsche"]
AF2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Load the most recent filtered accession file ===
filtered_files = sorted(PROCESSED_DIR.glob("accessions_AF2_lengthGT150_*.csv"))
if not filtered_files:
    raise FileNotFoundError(
        f"No filtered accession file found in: {PROCESSED_DIR}\n"
        "Run the Part 1 filtering step to generate "
        "'accessions_AF2_lengthGT150_*.csv'."
    )

filtered_path = filtered_files[-1]
print(f"Loading accession list from: {filtered_path.name}")

df = pd.read_csv(filtered_path)
if "AccessionID" not in df.columns:
    raise ValueError(
        f"'AccessionID' column not found in {filtered_path.name}. "
        f"Columns present: {list(df.columns)}"
    )

accessions = df["AccessionID"].astype(str).str.strip().tolist()

print(f"Accessions to process: {len(accessions):,}")
print(f"Saving PDB files to: {AF2_OUTPUT_DIR.resolve()}")

# === HTTP settings ===
HEADERS = {"User-Agent": "manuscript-pipeline/1.0"}

# Number of worker threads for parallel downloading.
# Start conservatively (ie 8-12)
MAX_WORKERS = 12

# How often to print a general processing update
PROCESS_REPORT_EVERY = 10000

# How often to print a successful download update
DOWNLOAD_REPORT_EVERY = 10000


def get_pdb_url_from_api(acc: str, session: requests.Session) -> str | None:
    """
    Query the AFDB prediction API for one accession and return the PDB URL.

    Parameters
    ----------
    acc : str
        UniProt accession ID.
    session : requests.Session
        Session object used for HTTP requests.

    Returns
    -------
    str | None
        PDB URL if available, otherwise None.
    """
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{acc}"

    try:
        r = session.get(api_url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            return None

        payload = r.json()

        # AF2DB usually returns a list containing one record.
        if isinstance(payload, list) and len(payload) > 0 and isinstance(payload[0], dict):
            rec = payload[0]
        elif isinstance(payload, dict):
            rec = payload
        else:
            return None

        # The API typically provides the PDB download URL under "pdbUrl".
        pdb_url = rec.get("pdbUrl")
        return pdb_url if pdb_url else None

    except (requests.RequestException, ValueError):
        return None


def download_file(url: str, out_path: Path, session: requests.Session, retries: int = 2) -> bool:
    """
    Download a file from URL to out_path with basic retry logic.

    Parameters
    ----------
    url : str
        File download URL.
    out_path : Path
        Destination file path.
    session : requests.Session
        Session object used for HTTP requests.
    retries : int
        Number of retries after the initial attempt.

    Returns
    -------
    bool
        True if download succeeded, False otherwise.
    """
    for attempt in range(retries + 1):
        try:
            r = session.get(url, headers=HEADERS, timeout=60)
            if r.status_code == 200 and r.content:
                out_path.write_bytes(r.content)
                return True
        except requests.RequestException:
            pass

        # Simple backoff before retrying
        time.sleep(1.5 * (attempt + 1))

    return False


def process_accession(acc: str) -> tuple[str, str]:
    """
    Process a single accession:
      1. Check whether the PDB file is already present.
      2. Query AFDB API for a PDB URL.
      3. Download the PDB if available.

    Parameters
    ----------
    acc : str
        UniProt accession ID.

    Returns
    -------
    tuple[str, str]
        (accession, status), where status is one of:
        - "downloaded"
        - "skipped"
        - "failed_no_url"
        - "failed_download"
    """
    out_file = AF2_OUTPUT_DIR / f"{acc}.pdb"

    # Skip if the file already exists and is non-empty
    if out_file.exists() and out_file.stat().st_size > 0:
        return acc, "skipped"

    # Use one session per worker task for reliable parallel behavior
    with requests.Session() as session:
        pdb_url = get_pdb_url_from_api(acc, session)
        if not pdb_url:
            return acc, "failed_no_url"

        ok = download_file(pdb_url, out_file, session, retries=2)
        if ok:
            return acc, "downloaded"
        return acc, "failed_download"


# === Parallel download loop ===
success = []
failed = []
skipped = []

# Threshold for the next "structures downloaded" progress message
next_download_report = DOWNLOAD_REPORT_EVERY

print(f"Starting parallel download with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_accession, acc): acc for acc in accessions}

    for idx, future in enumerate(as_completed(futures), start=1):
        acc, status = future.result()

        if status == "downloaded":
            success.append(acc)
        elif status == "skipped":
            skipped.append(acc)
        else:
            failed.append(acc)

        # Report every 10,000 successful downloads
        if len(success) >= next_download_report:
            print(f"{len(success):,} structures downloaded")
            next_download_report += DOWNLOAD_REPORT_EVERY

        # Occasional overall progress update, regardless of outcome
        if idx == 1 or idx % PROCESS_REPORT_EVERY == 0 or idx == len(accessions):
            print(
                f"[{idx:,}/{len(accessions):,}] "
                f"downloaded={len(success):,} "
                f"skipped={len(skipped):,} "
                f"failed={len(failed):,}"
            )

# === Write timestamped download log ===
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = AF2_OUTPUT_DIR / f"AF2_download_log_{timestamp}.txt"

with log_file.open("w") as f:
    f.write(f"Input accession file: {filtered_path.name}\n")
    f.write(f"Output directory: {AF2_OUTPUT_DIR.resolve()}\n")
    f.write(f"Parallel workers: {MAX_WORKERS}\n\n")

    f.write("Downloaded:\n")
    for acc in success:
        f.write(acc + "\n")

    f.write("\nSkipped (already present):\n")
    for acc in skipped:
        f.write(acc + "\n")

    f.write("\nFailed (no PDB URL or download failed):\n")
    for acc in failed:
        f.write(acc + "\n")

# === Final summary ===
print("\nDownload step complete.")
print(f"Downloaded = {len(success):,}")
print(f"Skipped    = {len(skipped):,}")
print(f"Failed     = {len(failed):,}")
print(f"Download log written to: {log_file}")
print(f"PDB files currently in folder: {len(list(AF2_OUTPUT_DIR.glob('*.pdb'))):,}")

## Part 3. Metal-Coordination Mining from AF2 Models

This step scans downloaded AlphaFold2 structures for candidate metal-binding sites defined by paired histidines (2-His motifs).
For each AF2 model (.pdb files):
- Histidine pairs are identified based on geometric criteria using backbone (N, O) and side-chain (NE2) atoms.
- For each His–His pair, a midpoint is computed and the nearest candidate metal-coordinating residue types are quantified by distance.
- Results are written to:
  - a summary table of all detected 2His sites, and
  - a list of structures with no qualifying His pairs.


In [ ]:
# === FUNCTIONS ===

def euclidean_distance(a, b):
    """Euclidean distance between two 3D points (iterables length 3)"""
    return ((a[0] - b[0])**2 + (a[1] - b[1])**2 + (a[2] - b[2])**2) ** 0.5


def extract_his_coordinates(structure):
    """
    Extract coordinates for HIS residues:
      - NE2 (side chain)
      - N and O (backbone)
    Keys are (chain_id, resseq) to avoid collisions across chains
    """
    his_ne2 = {}
    his_o = {}
    his_n = {}

    for model in structure:
        for chain in model:
            chain_id = chain.id
            for residue in chain:
                if residue.get_resname() == "HIS":
                    resseq = residue.get_id()[1]
                    key = (chain_id, resseq)
                    for atom in residue:
                        name = atom.get_name()
                        if name == "NE2":
                            his_ne2[key] = atom.get_coord()
                        elif name == "O":
                            his_o[key] = atom.get_coord()
                        elif name == "N":
                            his_n[key] = atom.get_coord()

    return his_ne2, his_o, his_n


def find_his_pairs(his_ne2, his_o, his_n, cutoff=4.0):
    """
    Find pairs of HIS residues using geometric criteria.
    Pairs are returned as list of ((chain, resi), (chain, resi)).
    Requires all needed atoms present for both residues
    """
    keys = sorted(set(his_ne2.keys()) & set(his_o.keys()) & set(his_n.keys()))
    pairs = []

    for idx_i in range(len(keys)):
        for idx_j in range(idx_i + 1, len(keys)):
            i = keys[idx_i]
            j = keys[idx_j]

            # Distances used in your original screening
            dist1 = euclidean_distance(his_ne2[i], his_ne2[j])
            dist2 = euclidean_distance(his_o[i], his_n[j])
            dist3 = euclidean_distance(his_n[i], his_o[j])

            if dist1 < cutoff and dist2 < cutoff and dist3 < cutoff:
                pairs.append((i, j))

    return pairs


def find_closest_residues(structure, his_pairs, his_ne2, protein_length, pdb_path):
    """
    For each HIS pair:
      - compute midpoint between NE2 atoms
      - compute closest distances from midpoint to key atoms of candidate residue types
      - determine HX residue (two residues after first HIS) when possible
    Returns a dict keyed by pair -> info
    """
    structure_id = Path(pdb_path).stem

    # Key atoms per residue type
    key_atoms = {
        "ASN": ["ND2", "OD1"],
        "GLN": ["NE2", "OE1"],         
        "CYS": ["SG"],
        "HIS": ["NE2", "ND1"],
        "MET": ["SD"],
        "TYR": ["OH"],
        "TRP": ["NE1"],
        "ASP": ["OD1", "OD2", "CB", "CG"],
        "GLU": ["OE1", "OE2", "CB", "CG", "CD"],
        "ALA": ["CB"],
        "GLY": ["CA"],
        "PHE": ["CD1", "CD2", "CG", "CE1", "CE2", "CZ"],
        "ILE": ["CG1", "CG2", "CD1"],
        "LEU": ["CG", "CD1", "CD2"],
        "VAL": ["CG1", "CG2"],
        "LYS": ["CG", "CD", "CE", "NZ"],
        "PRO": ["CB", "CG", "CD"],
        "SER": ["OG"],
        "THR": ["OG1", "CG2"],
        "ARG": ["CZ", "NE", "NH1", "NH2"],
    }

    residue_types = list(key_atoms.keys())
    output = {}

    for pair in his_pairs:
        (chainA, resiA), (chainB, resiB) = pair

        midpoint = (
            (his_ne2[pair[0]][0] + his_ne2[pair[1]][0]) / 2,
            (his_ne2[pair[0]][1] + his_ne2[pair[1]][1]) / 2,
            (his_ne2[pair[0]][2] + his_ne2[pair[1]][2]) / 2,
        )

        closest_distances = {res: float("inf") for res in residue_types}

        HX_res = "NA"
        for model in structure:
            for chain in model:
                if chain.id == chainA:
                    try:
                        HX_res = chain[(" ", resiA + 2, " ")].get_resname()
                    except Exception:
                        HX_res = "NA"

                for residue in chain:
                    resseq = residue.get_id()[1]
                    # skip the two His residues (by chain+resi)
                    if (chain.id, resseq) in [pair[0], pair[1]]:
                        continue

                    resname = residue.get_resname()
                    if resname not in key_atoms:
                        continue

                    for atom in residue:
                        if atom.get_name() in key_atoms[resname]:
                            dist = euclidean_distance(midpoint, atom.get_coord())
                            if dist < closest_distances[resname]:
                                closest_distances[resname] = dist

        output[pair] = {
            "structure_id": structure_id,
            "protein_length": protein_length,
            "midpoint": midpoint,
            "closest_residues": closest_distances,
            "HX_res": HX_res,
        }

    return output


def main_function(pdb_path):
    """Parse PDB files, find His pairs, compute distances and context features"""
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("pdb", pdb_path)

    # protein length (count residues)
    protein_length = len(list(structure.get_residues()))

    his_ne2, his_o, his_n = extract_his_coordinates(structure)
    his_pairs = find_his_pairs(his_ne2, his_o, his_n)

    if not his_pairs:
        return {}

    return find_closest_residues(structure, his_pairs, his_ne2, protein_length, pdb_path)


# === MAIN ===

# Input AF2 models (pdb files)
AF2_DIR = DIRS["4_retrieved_AF2modelsche"]

# Output directory for this step
OUT_DIR = DIRS["5_find_2His"]
OUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
out_hits_path = OUT_DIR / f"find_2His_sites_AF2_output_{timestamp}.csv"
out_none_path = OUT_DIR / f"no_pairs_his_output_{timestamp}.txt"

header = (
    "structure,chainA,hisA,chainB,hisB,protein_length,HXres,"
    "closest_ASN,closest_GLN,closest_CYS,closest_HIS,closest_MET,closest_TYR,closest_TRP,"
    "closest_ASP,closest_GLU,closest_ALA,closest_GLY,closest_ILE,closest_LEU,closest_VAL,"
    "closest_LYS,closest_PRO,closest_SER,closest_THR,closest_PHE,closest_ARG\n"
)

pdb_files = sorted(AF2_DIR.glob("*.pdb"))
print(f"Scanning {len(pdb_files)} PDB files in: {AF2_DIR.resolve()}")

processed = 0
hits = 0
no_pairs = 0

with out_hits_path.open("w") as output_file, out_none_path.open("w") as no_pairs_file:
    output_file.write(header)

    for pdb_path in pdb_files:
        processed += 1
        if processed % 10000 == 0 or processed == 1 or processed == len(pdb_files):
            print(f"[{processed}/{len(pdb_files)}] hits={hits} no_pairs={no_pairs}")

        output = main_function(str(pdb_path))

        if not output:
            no_pairs += 1
            no_pairs_file.write(pdb_path.stem + "\n")
            continue

        for pair, info in output.items():
            hits += 1
            (chainA, hisA), (chainB, hisB) = pair
            closest = info["closest_residues"]

            # write "inf" as blank if never found (optional)
            def fmt(x):
                return "" if (x == float("inf")) else str(x)

            line = (
                f"{info['structure_id']},{chainA},{hisA},{chainB},{hisB},"
                f"{info['protein_length']},{info['HX_res']},"
                f"{fmt(closest['ASN'])},{fmt(closest['GLN'])},{fmt(closest['CYS'])},"
                f"{fmt(closest['HIS'])},{fmt(closest['MET'])},{fmt(closest['TYR'])},"
                f"{fmt(closest['TRP'])},{fmt(closest['ASP'])},{fmt(closest['GLU'])},"
                f"{fmt(closest['ALA'])},{fmt(closest['GLY'])},{fmt(closest['ILE'])},"
                f"{fmt(closest['LEU'])},{fmt(closest['VAL'])},{fmt(closest['LYS'])},"
                f"{fmt(closest['PRO'])},{fmt(closest['SER'])},{fmt(closest['THR'])},"
                f"{fmt(closest['PHE'])},{fmt(closest['ARG'])}\n"
            )
            output_file.write(line)

print(f"Done.\nWrote hits table: {out_hits_path}\nWrote no-pairs list: {out_none_path}")


### Summary of Identified 2-His Metal-Binding Motifs

The step below summarizes the distribution of detected 2His sites across all analyzed AF2 structures from the step above.

Using a predefined distance cutoff to define a nearby third coordinating residue:
- The total number of identified 2His motifs is computed.
- Sites containing a proximal Asp or Glu residue (consistent with canonical 2His-1-carboxylate metal-binding motifs) are counted.
- Sites lacking Asp/Glu but containing a nearby Ala or Gly residue are quantified separately.

These counts provide a global overview of the prevalence of canonical and non-canonical metal-coordination motifs within the screened structural dataset.


In [ ]:
hits_csv = out_hits_path  
df_hits = pd.read_csv(hits_csv)

# Convert distance columns to numeric (blanks -> NaN)
dist_cols = [c for c in df_hits.columns if c.startswith("closest_")]
for c in dist_cols:
    df_hits[c] = pd.to_numeric(df_hits[c], errors="coerce")

total_sites = len(df_hits)

# Define categories
has_asp_glu = (df_hits["closest_ASP"] <= 6) | (df_hits["closest_GLU"] <= 6)

has_ala_gly = (
    (df_hits["closest_ASP"] > 5.5) &
    (df_hits["closest_GLU"] > 5.5) &
    (df_hits["closest_HIS"] > 4.0) &
    (df_hits["closest_CYS"] > 4.0) &
    (df_hits["closest_MET"] > 4.0) &
    (df_hits["closest_ASN"] > 4.0) &
    (df_hits["closest_GLN"] > 4.0) &
    (df_hits["closest_TYR"] > 4.0) &
    (df_hits["closest_TRP"] > 4.0) &
    (df_hits["HXres"].isin(["ALA", "GLY"])) &
    ((df_hits["closest_GLY"] < 7.0) | (df_hits["closest_ALA"] < 7.0))
)

count_2his_1aspglu = int(has_asp_glu.sum())
count_ala_gly_instead = int((~has_asp_glu & has_ala_gly).sum())

# Save identified radical halogenases accession IDs:

out_dir = DIRS["6_retrieved_halgogenases_from2HisSite"]
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

hal_csv = out_dir / f"radical_halogenase_candidates_{timestamp}.csv"

df_hits.loc[has_ala_gly, ["structure"]].drop_duplicates().rename(
    columns={"structure": "accession_id"}
).to_csv(hal_csv, index=False)

print(f"Saved radical halogenase candidate list to: {hal_csv}")

print(f"There are {total_sites} total 2His sites identified,")
print(f"of which {count_2his_1aspglu} are 2His-1Asp/Glu sites,")
print(f"and {count_ala_gly_instead} are 2His sites that lack a third metal-coordinating residue "
      f"and instead contain a nearby Ala/Gly (HxA/G motif).")

# Save the same summary to a text file
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
summary_txt = DIRS["5_find_2His"] / f"2His_site_summary_{timestamp}.txt"

with summary_txt.open("w") as f:
    f.write(f"Hits table: {Path(hits_csv).name}\n")
    f.write(f"There are {total_sites} total 2His sites identified,\n")
    f.write(f"of which {count_2his_1aspglu} are 2His-1Asp/Glu sites,\n")
    f.write(f"and {count_ala_gly_instead} are 2His sites that lack a third metal-coordinating residue "
            f"and instead contain a nearby Ala/Gly.\n")

print(f"Summary written to: {summary_txt}")


## Part 4. Initial Validation Using Known Radical Halogenases

As an internal control, a curated list of previously characterized radical halogenases is used to verify:
1) that these accessions are present in the screened structural dataset, and
2) whether they are recovered by the halogenase prioritization criteria.

Curated list:

- SyrB2 = Q9RBY6 <br>
- OocP = K7WEY, <br>
- BesD = G8XHD5 <br>
- HalD = A0A0F4XRB2 <br>
- WelO5 =A0A067YX61 <br>
- Adev = A0A1U8X168 <br>
- DAH = A0A6M3RHU8 <br>
- BtnX = A8LT50

In [ ]:
# ---- Known positive controls ----
known_halogenases = [
    "Q9RBY6", "K7WEY7", "G8XHD5", "A0A0F4XRB2",
    "A0A067YX61", "A0A1U8X168", "A0A6M3RHU8", "A8LT50"
]

# ---- Load latest 2-His dataset (if df_2His not already in memory) ----
if "df_2His" not in globals():
    hits_files = sorted(DIRS["5_find_2His"].glob("find_2His_sites_AF2_output_*.csv"))
    if not hits_files:
        raise FileNotFoundError(
            "No 2-His hits table found in 5_find_2His.\n"
            "Run Part 3 (metal-coordination mining) first."
        )
    hits_path = hits_files[-1]
    print(f"Loading 2-His hits table from: {hits_path.name}")
    df_2His = pd.read_csv(hits_path)

# Ensure 'structure' column exists
if "structure" not in df_2His.columns:
    raise ValueError(f"'structure' column not found in df_2His. Columns: {list(df_2His.columns)}")

present_in_initial = [x for x in known_halogenases if x in set(df_2His["structure"].astype(str))]
missing_in_initial = [x for x in known_halogenases if x not in set(df_2His["structure"].astype(str))]

print(f"Known halogenases present in initial 2-His dataset: {len(present_in_initial)}/{len(known_halogenases)}")
if missing_in_initial:
    print("Missing from initial dataset:", ", ".join(missing_in_initial))

# ---- Evaluate recovery in halogenase hits (df_hits) ----
if "df_hits" in globals():
    if "structure" not in df_hits.columns:
        raise ValueError(f"'structure' column not found in df_hits. Columns: {list(df_hits.columns)}")

    recovered = [x for x in known_halogenases if x in set(df_hits["structure"].astype(str))]
    not_recovered = [x for x in known_halogenases if x not in set(df_hits["structure"].astype(str))]

    print(f"Known halogenases identified by metal-coordination mining (in df_hits): {len(recovered)}/{len(known_halogenases)}")
    if recovered:
        print("Recovered:", ", ".join(recovered))
    if not_recovered:
        print("Not recovered:", ", ".join(not_recovered))
else:
    print("Note: df_hits is not defined yet. Run the halogenase prioritization step, then re-run this block.")

print("NOTE: Please be aware that InterPro and the AlphaFold Protein Structure Database are continuously updated, therefore, results from the internal validation step should be interpreted in the context of the dataset retrieved at the time of analysis, and comparisons with previous runs should be made accordingly.")

## Part 5. Retrieve and Parse UniProt Annotations for Putative Halogenases

To annotate candidate radical halogenases, the corresponding UniProt accession IDs are submitted by the user to the UniProt ID mapping / retrieval tool (`www.uniprot.org/id-mapping`). The resulting FASTA file (containing UniProt header annotations and amino acid sequences) should be named `idmapping.fasta` and placed in the `inputs/` directory. <br>

The following code reads the above file and parses into a structured dataframe containing:<br>
`accession`, `organism name`, `taxonomy ID`, `gene name` (if available), `functional description`, and the `protein sequence`.

The parsed annotation table is written to the `7_getUNIPROTannotations_IDmapping` output directory for downstream analyses.


In [ ]:
def parse_uniprot_fasta_to_df(fasta_path: Path) -> pd.DataFrame:
    """
    Parse a UniProt FASTA file into a dataframe.
    Extracts:
      - structure (UniProt accession)
      - function_annotation (protein name/description)
      - organism_scientific (OS=...)
      - organism_taxID (OX=...)
      - gene_ID (GN=..., if present)
      - aa_seq
    """
    records = []
    entry = None
    seq_lines = []

    # This regex is tolerant of optional GN and varying spacing
    header_re = re.compile(
        r"^>(?:sp|tr)\|(?P<acc>[^|]+)\|.*?\s(?P<desc>.*?)\sOS=(?P<os>.*?)(?:\sOX=(?P<ox>\d+))"
        r"(?:\sGN=(?P<gn>\S+))?"
    )

    with fasta_path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                # Save previous entry
                if entry is not None:
                    entry["aa_seq"] = "".join(seq_lines)
                    records.append(entry)

                # Start new entry
                entry = {
                    "structure": np.nan,
                    "function_annotation": np.nan,
                    "organism_scientific": np.nan,
                    "organism_taxID": np.nan,
                    "gene_ID": np.nan,
                }
                seq_lines = []

                m = header_re.search(line)
                if m:
                    entry["structure"] = m.group("acc") or np.nan
                    entry["function_annotation"] = m.group("desc") or np.nan
                    entry["organism_scientific"] = m.group("os") or np.nan
                    entry["organism_taxID"] = m.group("ox") or np.nan
                    entry["gene_ID"] = m.group("gn") or np.nan
                else:
                    # Keep the raw header if parsing fails (helps debugging)
                    entry["function_annotation"] = line[1:]

            else:
                seq_lines.append(line)

        # Save last entry
        if entry is not None:
            entry["aa_seq"] = "".join(seq_lines)
            records.append(entry)

    return pd.DataFrame.from_records(records, columns=[
        "structure", "function_annotation", "organism_taxID",
        "organism_scientific", "gene_ID", "aa_seq"
    ])


# === Input / Output paths ===
# User places UniProt FASTA here:
FASTA_INPUT = INPUT_DIR / "idmapping.fasta"

# Output written here (run-specific):
OUT_UNIPROT_DIR = DIRS["7_getUNIPROTannotations_IDmapping"]
OUT_UNIPROT_DIR.mkdir(parents=True, exist_ok=True)

if not FASTA_INPUT.exists():
    raise FileNotFoundError(
        f"UniProt FASTA input file not found:\n{FASTA_INPUT}\n\n"
        "Please place the downloaded UniProt ID-mapping FASTA file at this location "
        "(or update FASTA_INPUT to point to your file)."
    )

# === Parse FASTA and save ===
df_fasta = parse_uniprot_fasta_to_df(FASTA_INPUT)
print(df_fasta.head(3))

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = OUT_UNIPROT_DIR / f"putativeHals_UNIPROToutput_fasta_to_dataframe_{timestamp}.csv"
df_fasta.to_csv(out_csv, index=False)

print(f"Saved parsed UniProt annotations table to: {out_csv}")
print(f"Parsed entries: {df_fasta.shape[0]}")


### Merge Structural Hits with UniProt Annotations and Apply Keyword-Based Filtering

Candidate hits (2His site table) are merged with UniProt-derived annotations to provide organism and functional context.
To remove obvious false positives (e.g., transcriptional regulators, globins, SAM enzymes), a keyword-based filter is applied to the UniProt functional description.
The filtered annotated table and a list of retained accession IDs are written to the `7_getUNIPROTannotations_IDmapping` output directory.


In [ ]:
# === Load latest pipeline outputs ===
HALS_DIR = DIRS["6_retrieved_halgogenases_from2HisSite"]
UNIPROT_DIR = DIRS["7_getUNIPROTannotations_IDmapping"]

his_files = sorted(HALS_DIR.glob("*.csv"))
uniprot_files = sorted(UNIPROT_DIR.glob("putativeHals_UNIPROToutput_fasta_to_dataframe_*.csv"))

if not his_files:
    raise FileNotFoundError(f"No 2-His halogenase CSV found in: {HALS_DIR}")
if not uniprot_files:
    raise FileNotFoundError(f"No UniProt annotation CSV found in: {UNIPROT_DIR}")

df_2His_path = his_files[-1]
df_uniprot_path = uniprot_files[-1]

print(f"Loading 2-His putative halogenases from: {df_2His_path.name}")
print(f"Loading UniProt annotations from: {df_uniprot_path.name}")

df_2His = pd.read_csv(df_2His_path)
df_uniprot = pd.read_csv(df_uniprot_path)
df_uniprot = df_uniprot.rename(columns={"structure": "accession_id"})

# === Merge ===
df_merged = pd.merge(df_2His, df_uniprot, on="accession_id", how="left")
print("Merged rows:", len(df_merged))

# === Keyword-based filtering ===
functions_to_remove = [
    "transcription", "regulator", "AraC", "globin", "AlkB",
    "glutelin", "TehB", "tet", "fragment", "chemotaxis",
    "helix-turn-helix", "tellurite", "adenosyl", "SAM", "ferredoxin"
]

pattern = "|".join(functions_to_remove)

filtered_df = df_merged[
    ~df_merged["function_annotation"].astype(str)
    .str.contains(pattern, case=False, na=False)
].copy()

print("Filtered rows:", len(filtered_df))

# === Save outputs ===
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

out_ids = UNIPROT_DIR / f"putativeHals_annotated_and_filtered_IDsONLY_{timestamp}.csv"
out_full = UNIPROT_DIR / f"putativeHals_annotated_and_filtered_{timestamp}.csv"

filtered_df[["accession_id"]].drop_duplicates().to_csv(out_ids, index=False)
filtered_df.to_csv(out_full, index=False)

print(f"Saved IDs-only list to: {out_ids}")
print(f"Saved full annotated+filtered table to: {out_full}")


## Map Identified Halogenases to InterPro Families (Supplementary Table 1)

To generate *Supplementary Table 1*, accession IDs from the final SSN analysis (all / known-cluster / new-cluster halogenases)
are mapped back to their originating InterPro/PFAM family IDs using the compiled accession-to-family table from Part 1. <br>
Counts per family are merged into the *Supplementary Table 1* template and exported as a new CSV.


In [ ]:
# Map SSN halogenase accessions to InterPro/PFAM families (Supplementary Table 1)
#
# User-provided inputs (place directly in INPUT_DIR):
#   1) accession_IDs_in_SSN.csv
#        columns required: all_hals, known_hals, new_hals
#   2) Table_S1_proteinFamilies.csv
#        column required: Accession ID   (InterPro/PFAM family IDs)
#
# Pipeline-generated input (from outputs of this run):
#   - latest compiled_accession_IDs_*.csv in DIRS["3_processed_accessionIDs"]
#        columns required: AccessionID, family_ID
#
# Output:
#   - OUTDIR/8_TableS1/Table_S1_proteinFamilies_countsadded_<timestamp>.csv

# Output folder
OUT_TABLES1 = OUTDIR / "8_TableS1"
OUT_TABLES1.mkdir(parents=True, exist_ok=True)

# User-provided inputs
SSN_IDS_FILE      = INPUT_DIR / "accession_IDs_in_SSN.csv"
TABLE_S1_TEMPLATE = INPUT_DIR / "Table_S1_proteinFamilies.csv"

missing_user = [p for p in [SSN_IDS_FILE, TABLE_S1_TEMPLATE] if not p.exists()]
if missing_user:
    raise FileNotFoundError(
        "Missing required user-provided input(s) in INPUT_DIR:\n"
        + "\n".join(str(p) for p in missing_user)
    )

# Pipeline-generated input (from run outputs)
PROCESSED_DIR = DIRS["3_processed_accessionIDs"]
compiled_files = sorted(PROCESSED_DIR.glob("compiled_accession_IDs_*.csv"))
if not compiled_files:
    raise FileNotFoundError(
        f"No compiled accession mapping file found in: {PROCESSED_DIR}\n"
        "Run the accession compilation step first (Part 3 compile step)."
    )
COMPILED_MAP_FILE = compiled_files[-1]  # latest

# Load inputs
df_map = pd.read_csv(COMPILED_MAP_FILE)
df_ssn = pd.read_csv(SSN_IDS_FILE)
df_table = pd.read_csv(TABLE_S1_TEMPLATE)

# Quality checks
required_map_cols = {"AccessionID", "family_ID"}
if not required_map_cols.issubset(df_map.columns):
    raise ValueError(
        f"{COMPILED_MAP_FILE.name} must contain columns {required_map_cols}. "
        f"Found: {list(df_map.columns)}"
    )

for col in ["all_hals", "known_hals", "new_hals"]:
    if col not in df_ssn.columns:
        raise ValueError(
            f"{SSN_IDS_FILE.name} must contain column '{col}'. Found: {list(df_ssn.columns)}"
        )

if "Accession ID" not in df_table.columns:
    raise ValueError(
        f"{TABLE_S1_TEMPLATE.name} must contain column 'Accession ID'. Found: {list(df_table.columns)}"
    )

# Get counts per family for a given SSN column 
def family_counts_for_column(df_ids: pd.DataFrame, id_col: str, df_map: pd.DataFrame) -> pd.DataFrame:
    ids = df_ids[[id_col]].dropna().copy()
    ids[id_col] = ids[id_col].astype(str).str.strip()

    merged = ids.merge(
        df_map[["AccessionID", "family_ID"]],
        left_on=id_col,
        right_on="AccessionID",
        how="left"
    ).dropna(subset=["family_ID"])

    counts = (
        merged.groupby("family_ID")[id_col]
        .count()
        .reset_index()
        .rename(columns={"family_ID": "Accession ID", id_col: f"count_{id_col}"})
    )
    return counts

# Compute counts 
counts_all   = family_counts_for_column(df_ssn, "all_hals", df_map)
counts_known = family_counts_for_column(df_ssn, "known_hals", df_map)
counts_new   = family_counts_for_column(df_ssn, "new_hals", df_map)

# Merge counts into Supplementary Table 1 
df_table_counts = (
    df_table
    .merge(counts_all, on="Accession ID", how="left")
    .merge(counts_known, on="Accession ID", how="left")
    .merge(counts_new, on="Accession ID", how="left")
)

for c in ["count_all_hals", "count_known_hals", "count_new_hals"]:
    df_table_counts[c] = df_table_counts[c].fillna(0).astype(int)

# Export 
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = OUT_TABLES1 / f"Table_S1_proteinFamilies_countsadded_{timestamp}.csv"
df_table_counts.to_csv(out_csv, index=False)

print("Loaded:")
print(" - Pipeline mapping:", COMPILED_MAP_FILE)
print(" - User SSN file:", SSN_IDS_FILE)
print(" - User Table S1 template:", TABLE_S1_TEMPLATE)
print("\nWrote:")
print(" -", out_csv)


### Part 5. Classification of 2His Sites by Third-Sphere Coordination

Moving beyond the targeted discovery of radical halogenases to expore the diversity and distribution of 2His-Xn metal coordinating motifs in the cupin superfamily, where X includes other potentially coordinating side chains (His, Asp, Glu, Asn, Gln, Cys, Met, Tyr) and n = 0-4 to include coordination geometries from the bi-, tri-, tetra-, penta-, and hexacoordinate mononuclear metal sites.

Then, rather than focusing only on canonical facial triads (2His-1Asp/Glu) or single alternative ligands (2His only), this analysis systematically classifies each site according to the total number of additional coordinating residues, resulting in the folloowing categories, where X can include any combination of His, Asp, Glu, Asn, Gln, Cys, Met, Tyr:

- **2His–0X**
- **2His–1X**
- **2His–2X**
- **2His–3X**
- **2His–4X**

This provides a global view of the diversity of metal coordination geometries present in the dataset and allows comparison between canonical and non-canonical active-site architectures.

Finally, to enable downstream SSN visualization at manageable size, we generate a **stratified subsample** of proteins that preserves motif diversity: all motif bins with ≥10 proteins are retained, and up to ~2,000 total sequences are sampled across bins. The resulting SSN input file contains only two columns (`accession_id`, `motif`). This is shown in **Extended Data Figure 10** of the manuscript.


In [ ]:
# Metal-coordination mining (coordination-sphere only)
# - Finds 2-His sites from the subset of AF2 structures
# - For each site, collects ALL candidate ligand residues within 4.5 Å
#   (distance computed from midpoint to *coordinating/key atoms* only)
# - Asp/Glu are treated like other residues: ONLY OD/OE atoms (no backbone/CB/CG scan)
# - Writes outputs to PROJECT_ROOT:
#     1) 2his_sites_all_<timestamp>.csv
#     2) 2his_site_ligands_within4p5A_<timestamp>.csv
#     3) no_pairs_<timestamp>.txt

from pathlib import Path
import datetime as dt
import pandas as pd
import numpy as np
from Bio.PDB import PDBParser

# User-facing settings
HIS_PAIR_CUTOFF = 4.0      # Å: NE2-NE2 and backbone cross checks
LIGAND_CUTOFF   = 4.5      # Å: ONLY store ligands within this distance
HX_OFFSET       = 2        # residue offset for HX check (HisA + 2)
QUIET_PDB       = True

# Hard-code your PDB directory here:
AF2_DIR = Path("your_path_to_folder").expanduser().resolve() #<-- ADD YOUR PATH TO THE STRUCTURE FOLDER
if not AF2_DIR.exists():
    raise FileNotFoundError(f"PDB directory not found: {AF2_DIR}")

# Output files saved in PROJECT_ROOT
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_SITES   = PROJECT_ROOT / f"2his_sites_all_{timestamp}.csv"
OUT_LIGANDS = PROJECT_ROOT / f"2his_site_ligands_within{str(LIGAND_CUTOFF).replace('.','p')}A_{timestamp}.csv"
OUT_NOPAIRS = PROJECT_ROOT / f"no_pairs_{timestamp}.txt"

print("Scanning PDBs in:", AF2_DIR)
print("Writing site table to:", OUT_SITES)
print("Writing ligand table to:", OUT_LIGANDS)
print("Writing no-pairs list to:", OUT_NOPAIRS)

# -------------------------
# Key atoms for metal coordination
# -------------------------
KEY_ATOMS = {
    "ASN": ["ND2", "OD1"],
    "GLN": ["NE2", "OE1"],
    "CYS": ["SG"],
    "HIS": ["NE2", "ND1"],
    "MET": ["SD"],
    "TYR": ["OH"],
    "TRP": ["NE1"],
    "ASP": ["OD1", "OD2"],
    "GLU": ["OE1", "OE2"],
    # Optional extra sidechains (kept for context; not metal ligands)
    "PHE": ["CD1", "CD2", "CE1", "CE2", "CZ"],
    "ILE": ["CG1", "CG2", "CD1"],
    "LEU": ["CG", "CD1", "CD2"],
    "VAL": ["CG1", "CG2"],
    "LYS": ["NZ"],
    "PRO": ["CB", "CG", "CD"],
    "SER": ["OG"],
    "THR": ["OG1"],
    "ARG": ["NE", "NH1", "NH2"],
}
RES_TYPES = list(KEY_ATOMS.keys())

# Helper functions
def euclidean_distance(a, b) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b))

def extract_his_coordinates(structure):
    """
    Extract coordinates for HIS residues:
      - NE2 (side chain)
      - N and O (backbone)
    Keys are (chain_id, resseq).
    """
    his_ne2, his_o, his_n = {}, {}, {}

    for model in structure:
        for chain in model:
            chain_id = chain.id
            for residue in chain:
                if residue.get_resname() != "HIS":
                    continue
                resseq = residue.get_id()[1]
                key = (chain_id, resseq)

                for atom in residue:
                    name = atom.get_name()
                    if name == "NE2":
                        his_ne2[key] = atom.get_coord()
                    elif name == "O":
                        his_o[key] = atom.get_coord()
                    elif name == "N":
                        his_n[key] = atom.get_coord()

    return his_ne2, his_o, his_n

def find_his_pairs(his_ne2, his_o, his_n, cutoff=4.0):
    """
    Find pairs of HIS residues using geometric criteria.
    Returns list of ((chain, resi), (chain, resi)).
    """
    keys = sorted(set(his_ne2) & set(his_o) & set(his_n))
    pairs = []

    for i_idx in range(len(keys)):
        for j_idx in range(i_idx + 1, len(keys)):
            i = keys[i_idx]
            j = keys[j_idx]

            dist1 = euclidean_distance(his_ne2[i], his_ne2[j])  # NE2-NE2
            dist2 = euclidean_distance(his_o[i],  his_n[j])      # O(i) - N(j)
            dist3 = euclidean_distance(his_n[i],  his_o[j])      # N(i) - O(j)

            if dist1 < cutoff and dist2 < cutoff and dist3 < cutoff:
                pairs.append((i, j))

    return pairs

def _safe_get_resname(chain, resseq: int) -> str:
    try:
        return chain[(" ", int(resseq), " ")].get_resname()
    except Exception:
        return "NA"

def collect_ligands_within_cutoff(structure, midpoint, his_pair_keys, cutoff=4.5):
    """
    Collect ALL residues whose key atoms are within cutoff of midpoint.
    Returns:
      - ligand_rows: list[dict] (one per ligand residue found)
      - mins_counts: dict {RES: (min_dist, count_within_cutoff)}
    """
    min_dist = {res: np.inf for res in RES_TYPES}
    count    = {res: 0 for res in RES_TYPES}
    ligand_rows = []

    for model in structure:
        for chain in model:
            chain_id = chain.id
            for residue in chain:
                resname = residue.get_resname()
                if resname not in KEY_ATOMS:
                    continue

                resseq = residue.get_id()[1]
                if (chain_id, resseq) in his_pair_keys:
                    continue

                best_atom = None
                best_d = np.inf

                for atom in residue:
                    if atom.get_name() in KEY_ATOMS[resname]:
                        d = euclidean_distance(midpoint, atom.get_coord())
                        if d < best_d:
                            best_d = d
                            best_atom = atom.get_name()

                if best_atom is None:
                    continue

                if best_d <= cutoff:
                    count[resname] += 1
                    if best_d < min_dist[resname]:
                        min_dist[resname] = best_d

                    ligand_rows.append({
                        "ligand_chain": chain_id,
                        "ligand_resseq": int(resseq),
                        "ligand_resname": resname,
                        "ligand_best_atom": best_atom,
                        "ligand_min_dist": float(best_d),
                    })

    mins_counts = {res: (float(min_dist[res]), int(count[res])) for res in RES_TYPES}
    return ligand_rows, mins_counts

def analyze_structure(pdb_path: Path, parser: PDBParser):
    """
    Returns:
      - site_rows: list[dict] (one per 2-His site)
      - ligand_rows: list[dict] (one per ligand residue per site)
      - had_any_pairs: bool
    """
    structure = parser.get_structure("pdb", str(pdb_path))
    protein_length = len(list(structure.get_residues()))
    structure_id = pdb_path.stem

    his_ne2, his_o, his_n = extract_his_coordinates(structure)
    his_pairs = find_his_pairs(his_ne2, his_o, his_n, cutoff=HIS_PAIR_CUTOFF)

    if not his_pairs:
        return [], [], False

    site_rows = []
    ligand_rows_all = []

    for (chainA, resiA), (chainB, resiB) in his_pairs:
        # midpoint between NE2 atoms
        midpoint = (
            (his_ne2[(chainA, resiA)][0] + his_ne2[(chainB, resiB)][0]) / 2,
            (his_ne2[(chainA, resiA)][1] + his_ne2[(chainB, resiB)][1]) / 2,
            (his_ne2[(chainA, resiA)][2] + his_ne2[(chainB, resiB)][2]) / 2,
        )

        # HX residue (resiA + 2) in chainA
        HXres = "NA"
        for model in structure:
            for chain in model:
                if chain.id == chainA:
                    HXres = _safe_get_resname(chain, int(resiA) + HX_OFFSET)
                    break

        his_pair_keys = {(chainA, resiA), (chainB, resiB)}
        lig_rows, mins_counts = collect_ligands_within_cutoff(
            structure, midpoint, his_pair_keys, cutoff=LIGAND_CUTOFF
        )

        # attach site identifiers to each ligand row
        for r in lig_rows:
            r.update({
                "structure": structure_id,
                "chainA": chainA,
                "hisA": int(resiA),
                "chainB": chainB,
                "hisB": int(resiB),
                "protein_length": int(protein_length),
                "HXres": HXres,
            })
        ligand_rows_all.extend(lig_rows)

        # build one site row with mins + counts per residue type
        row = {
            "structure": structure_id,
            "chainA": chainA, "hisA": int(resiA),
            "chainB": chainB, "hisB": int(resiB),
            "protein_length": int(protein_length),
            "HXres": HXres,
        }
        for res in RES_TYPES:
            mn, ct = mins_counts[res]
            row[f"min_{res}"] = (np.nan if np.isinf(mn) else mn)
            row[f"n_{res}_within{str(LIGAND_CUTOFF).replace('.','p')}A"] = ct

        site_rows.append(row)

    return site_rows, ligand_rows_all, True

# Run over PDB files
pdb_files = sorted(AF2_DIR.glob("*.pdb"))
print(f"\nFound {len(pdb_files)} PDB files.")

parser = PDBParser(QUIET=QUIET_PDB)

all_site_rows = []
all_ligand_rows = []
no_pairs = []
failed = []

for i, pdb_path in enumerate(pdb_files, start=1):
    if i == 1 or i % 50000 == 0 or i == len(pdb_files):
        print(f"[{i}/{len(pdb_files)}] {pdb_path.name}")

    try:
        site_rows, lig_rows, had_pairs = analyze_structure(pdb_path, parser)
    except Exception as e:
        failed.append(f"{pdb_path.name}\t{repr(e)}")
        no_pairs.append(pdb_path.stem)
        continue

    if not had_pairs:
        no_pairs.append(pdb_path.stem)
        continue

    all_site_rows.extend(site_rows)
    all_ligand_rows.extend(lig_rows)

# Write outputs
df_sites = pd.DataFrame(all_site_rows)
df_ligs  = pd.DataFrame(all_ligand_rows)

df_sites.to_csv(OUT_SITES, index=False)
df_ligs.to_csv(OUT_LIGANDS, index=False)
OUT_NOPAIRS.write_text("\n".join(no_pairs) + ("\n" if no_pairs else ""))

print("\nDone.")
print("Total 2-His sites:", len(df_sites))
print("Proteins with 2-His sites:", df_sites["structure"].nunique() if not df_sites.empty else 0)
print("Ligand rows (≤ cutoff):", len(df_ligs))
print("Wrote:", OUT_SITES)
print("Wrote:", OUT_LIGANDS)
print("Wrote:", OUT_NOPAIRS)

if failed:
    out_failed = PROJECT_ROOT / f"parse_failures_{timestamp}.txt"
    out_failed.write_text("\n".join(failed) + "\n")
    print("Wrote parse failures:", out_failed)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import datetime as dt

# Coordination classification using the mining outputs (from step above)
#   - Reads latest 2his_sites_all_*.csv that contains:
#       min_<AA> and n_<AA>_within4p5A columns
#   - Uses the n_* counts (handles multiple same-type ligands)
#   - Produces:
#       (1) site-level classification CSV (with class + motif labels)
#       (2) summary CSV (sites/proteins per class/motif)
#   - Saves outputs in PROJECT_ROOT
#

# 1) Load latest 2his_sites_all_*.csv
pattern = "2his_sites_all_*.csv"
candidates = list(PROJECT_ROOT.glob(pattern)) + list((PROJECT_ROOT / "outputs").rglob(pattern))
candidates = sorted(set(candidates))

if not candidates:
    raise FileNotFoundError(f"No files matching {pattern} found under: {PROJECT_ROOT}")

def _ts(p: Path) -> str:
    m = re.search(r"(\d{8}_\d{6})", p.name)
    return m.group(1) if m else ""

timestamped = [p for p in candidates if _ts(p)]
if timestamped:
    candidates_sorted = sorted(timestamped, key=lambda p: _ts(p))
else:
    candidates_sorted = sorted(candidates, key=lambda p: p.stat().st_mtime)

HITS_FILE = candidates_sorted[-1]
print("Using hits file:", HITS_FILE)

df = pd.read_csv(HITS_FILE)
print(f"Loaded {len(df)} rows")
print("Columns:", list(df.columns)[:15], "...")

# 2) Classification settings
X_LIGANDS = ["HIS", "ASP", "GLU", "ASN", "GLN", "CYS", "MET", "TYR"]

count_cols = [c for c in df.columns if c.startswith("n_") and "_within" in c]
if not count_cols:
    raise ValueError("No n_*_within* columns found")

m = re.search(r"_within(.+)$", count_cols[0])
WITHIN_SUFFIX = m.group(1) if m else "4p5A"   
print("Detected within-suffix for counts:", WITHIN_SUFFIX)

def ncol(aa: str) -> str:
    return f"n_{aa}_within{WITHIN_SUFFIX}"

def mincol(aa: str) -> str:
    return f"min_{aa}"


# 3) Validate required columns

required = ["structure"] + [ncol(aa) for aa in X_LIGANDS]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in {HITS_FILE.name}: {missing}")

# Coerce numeric
for aa in X_LIGANDS:
    df[ncol(aa)] = pd.to_numeric(df[ncol(aa)], errors="coerce").fillna(0).astype(int)
    if mincol(aa) in df.columns:
        df[mincol(aa)] = pd.to_numeric(df[mincol(aa)], errors="coerce")

# 4) Compute:
#    - per-type counts 
#    - total number of X ligands (sum of counts)
#    - bin class: 2His-0X/1X/2X/3X/4X (cap >=4)
#    - motif labels:
#        * "combo": unique residue types present (no multiplicity)
#        * "motif_mult": includes multiplicity e.g. HISx4+GLUx1

# Total number of X ligands (with multiplicity)
df["n_X"] = df[[ncol(aa) for aa in X_LIGANDS]].sum(axis=1).astype(int)

# Bin class by n_X (cap at 4X)
df["n_X_capped"] = df["n_X"].clip(upper=4)

def class_label(n: int) -> str:
    if n <= 0:
        return "2His-0X"
    if n >= 4:
        return "2His-4X"
    return f"2His-{n}X"

df["class"] = df["n_X_capped"].apply(class_label)

# Presence (no multiplicity)
for aa in X_LIGANDS:
    df[f"has_{aa}"] = df[ncol(aa)] > 0

def combo_label(row) -> str:
    present = [aa for aa in X_LIGANDS if row[f"has_{aa}"]]
    return "+".join(present) if present else "NONE"

df["combo"] = df.apply(combo_label, axis=1)

# Motif label with multiplicity
def motif_mult_label(row) -> str:
    parts = []
    for aa in X_LIGANDS:
        k = int(row[ncol(aa)])
        if k > 0:
            parts.append(f"{aa}x{k}")
    return "+".join(parts) if parts else "NONE"

df["motif_mult"] = df.apply(motif_mult_label, axis=1)

# 5) Summary prints

total_sites = len(df)
total_proteins = df["structure"].nunique()

print("\n=== 2-His site coordination summary (4.5 Å sphere) ===")
print(f"Sites: {total_sites}")
print(f"Proteins: {total_proteins}")

print("\nSites by class:")
print(df["class"].value_counts().sort_index())

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

print("\nTop motifs (by sites) [unique residue types]:")
print(df["combo"].value_counts().head(50))

print("\nTop motifs (by sites) [with multiplicity]:")
print(df["motif_mult"].value_counts().head(50))

# 6) Write outputs to PROJECT_ROOT

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

out_sites = PROJECT_ROOT / f"2His_coordination_classification_{timestamp}.csv"
df.to_csv(out_sites, index=False)

summary = (
    df.groupby(["class", "combo", "motif_mult"])
      .agg(
          sites=("motif_mult", "size"),
          proteins=("structure", "nunique"),
          mean_nX=("n_X", "mean"),
      )
      .reset_index()
      .sort_values(["class", "sites"], ascending=[True, False])
)

out_summary = PROJECT_ROOT / f"2His_coordination_summary_{timestamp}.csv"
summary.to_csv(out_summary, index=False)

print("\nWrote site-level classification:", out_sites)
print("Wrote summary table:", out_summary)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import datetime as dt

# Re-analyze 2His-X motifs across a series of stricter cutoffs
#
#   NONE = no residues from the candidate coordinating set
#          {HIS, ASP, GLU, ASN, GLN, CYS, MET, TYR}
#          within the given cutoff.
#
# IMPORTANT:
#   This code reads BOTH:
#   (1) the full 2-His site table  -> 2his_sites_all_*.csv
#   (2) the ligand-detail table    -> 2his_site_ligands_within4p5A_*.csv
#
#   The full site table defines the complete universe of 2-His sites.
#   The ligand-detail table is used only to assign per-cutoff ligand counts.
#
# OUTPUTS:
#   1) Site-level table with motif columns for each cutoff:
#        2His_coordination_sites_multicutoff_<timestamp>.csv
#   2) Motif summary table (merged across cutoffs):
#        2His_motif_counts_multicutoff_<timestamp>.csv

# User settings
CUTOFFS = [3.0, 3.5, 4.0, 4.5]
X_LIGANDS = ["HIS", "ASP", "GLU", "ASN", "GLN", "CYS", "MET", "TYR"]

# Helper for timestamp sorting
def extract_timestamp(p: Path) -> str:
    m = re.search(r"(\d{8}_\d{6})", p.name)
    return m.group(1) if m else ""

# 1) Locate latest full site table
site_pattern = "2his_sites_all_*.csv"
site_candidates = list(PROJECT_ROOT.glob(site_pattern)) + list(PROJECT_ROOT.rglob(site_pattern))
site_candidates = sorted(set(site_candidates))

if not site_candidates:
    raise FileNotFoundError(f"No files matching {site_pattern} found under:\n{PROJECT_ROOT}")

site_timestamped = [p for p in site_candidates if extract_timestamp(p)]
if site_timestamped:
    site_candidates = sorted(site_timestamped, key=lambda p: extract_timestamp(p))
else:
    site_candidates = sorted(site_candidates, key=lambda p: p.stat().st_mtime)

SITE_FILE = site_candidates[-1]
print("Using full site table:", SITE_FILE)

# 2) Locate latest ligand-detail table
lig_pattern = "2his_site_ligands_within4p5A_*.csv"
lig_candidates = list(PROJECT_ROOT.glob(lig_pattern)) + list(PROJECT_ROOT.rglob(lig_pattern))
lig_candidates = sorted(set(lig_candidates))

if not lig_candidates:
    raise FileNotFoundError(f"No files matching {lig_pattern} found under:\n{PROJECT_ROOT}")

lig_timestamped = [p for p in lig_candidates if extract_timestamp(p)]
if lig_timestamped:
    lig_candidates = sorted(lig_timestamped, key=lambda p: extract_timestamp(p))
else:
    lig_candidates = sorted(lig_candidates, key=lambda p: p.stat().st_mtime)

LIG_FILE = lig_candidates[-1]
print("Using ligand-detail table:", LIG_FILE)

# 3) Load data
df_sites_full = pd.read_csv(SITE_FILE)
df_ligs = pd.read_csv(LIG_FILE)

print(f"Loaded full site table: {len(df_sites_full)} rows")
print(f"Loaded ligand-detail table: {len(df_ligs)} rows")

# 4) Normalize / validate columns
# Fix occasional typo if present;
if "structure" not in df_sites_full.columns and "tructure" in df_sites_full.columns:
    df_sites_full = df_sites_full.rename(columns={"tructure": "structure"})

required_site_cols = ["structure", "chainA", "hisA", "chainB", "hisB"]
missing_site = [c for c in required_site_cols if c not in df_sites_full.columns]
if missing_site:
    raise ValueError(f"Missing required columns in full site table: {missing_site}")

required_lig_cols = [
    "structure", "chainA", "hisA", "chainB", "hisB",
    "ligand_resname", "ligand_min_dist"
]
missing_lig = [c for c in required_lig_cols if c not in df_ligs.columns]
if missing_lig:
    raise ValueError(f"Missing required columns in ligand-detail table: {missing_lig}")

df_ligs["ligand_min_dist"] = pd.to_numeric(df_ligs["ligand_min_dist"], errors="coerce")

site_keys = ["structure", "chainA", "hisA", "chainB", "hisB"]

# Base site universe = ALL 2-His sites
base_cols = [c for c in ["structure", "chainA", "hisA", "chainB", "hisB", "protein_length", "HXres"] if c in df_sites_full.columns]
df_sites = df_sites_full[base_cols].drop_duplicates().copy()

# Ensure key columns have compatible dtypes for merging
for col in ["hisA", "hisB"]:
    if col in df_sites.columns:
        df_sites[col] = pd.to_numeric(df_sites[col], errors="coerce").astype("Int64")
    if col in df_ligs.columns:
        df_ligs[col] = pd.to_numeric(df_ligs[col], errors="coerce").astype("Int64")

# 5) Build per-cutoff site-level annotations
for cutoff in CUTOFFS:
    cutoff_label = f"{cutoff:.1f}".replace(".", "p") + "A"
    print(f"\nProcessing cutoff {cutoff:.1f} Å ({cutoff_label})")

    # Restrict ligand rows to X-ligands only AND cutoff
    df_cut = df_ligs[
        df_ligs["ligand_resname"].isin(X_LIGANDS) &
        df_ligs["ligand_min_dist"].notna() &
        (df_ligs["ligand_min_dist"] <= cutoff)
    ].copy()

    # Count residues of each ligand type per site
    for aa in X_LIGANDS:
        sub = df_cut[df_cut["ligand_resname"] == aa]

        count_col = f"n_{aa}_within{cutoff_label}"
        min_col = f"min_{aa}_{cutoff_label}"

        counts = (
            sub.groupby(site_keys)
               .size()
               .reset_index(name=count_col)
        )

        mins = (
            sub.groupby(site_keys)["ligand_min_dist"]
               .min()
               .reset_index(name=min_col)
        )

        df_sites = df_sites.merge(counts, on=site_keys, how="left")
        df_sites = df_sites.merge(mins, on=site_keys, how="left")

        df_sites[count_col] = df_sites[count_col].fillna(0).astype(int)

    # Total number of X ligands at this cutoff
    count_cols = [f"n_{aa}_within{cutoff_label}" for aa in X_LIGANDS]
    df_sites[f"n_X_{cutoff_label}"] = df_sites[count_cols].sum(axis=1).astype(int)

    # Total multiplicity INCLUDING the base 2 His
    df_sites[f"total_multiplicity_{cutoff_label}"] = 2 + df_sites[f"n_X_{cutoff_label}"]

    # Coordination class
    def class_label(n):
        if n <= 0:
            return "2His-0X"
        if n >= 4:
            return "2His-4X"
        return f"2His-{n}X"

    df_sites[f"coord_class_{cutoff_label}"] = df_sites[f"n_X_{cutoff_label}"].apply(class_label)

    # Presence-only motif
    def combo_label(row):
        present = [aa for aa in X_LIGANDS if row[f"n_{aa}_within{cutoff_label}"] > 0]
        return "+".join(present) if present else "NONE"

    df_sites[f"combo_{cutoff_label}"] = df_sites.apply(combo_label, axis=1)

    # Multiplicity-aware motif
    def motif_mult_label(row):
        parts = []
        for aa in X_LIGANDS:
            n = int(row[f"n_{aa}_within{cutoff_label}"])
            if n > 0:
                parts.append(f"{aa}x{n}")
        return "+".join(parts) if parts else "NONE"

    df_sites[f"motif_mult_{cutoff_label}"] = df_sites.apply(motif_mult_label, axis=1)

    # Quick sanity print
    none_count = int((df_sites[f"motif_mult_{cutoff_label}"] == "NONE").sum())
    print(f"  Total sites: {len(df_sites)}")
    print(f"  NONE sites at {cutoff:.1f} Å: {none_count}")
    print(f"  Top motifs:")
    print(df_sites[f"motif_mult_{cutoff_label}"].value_counts().head(10))

# 6) Build merged motif summary table across cutoffs
#    (site counts only; one row per motif)
tables = []

for cutoff in CUTOFFS:
    cutoff_label = f"{cutoff:.1f}".replace(".", "p") + "A"

    motif_counts = (
    df_sites[f"motif_mult_{cutoff_label}"]
    .value_counts()
    .rename_axis("motif")
    .reset_index(name=f"sites_{cutoff_label}"))

    tables.append(motif_counts)

df_final = tables[0]
for t in tables[1:]:
    df_final = df_final.merge(t, on="motif", how="outer")

df_final = df_final.fillna(0)

site_cols = [c for c in df_final.columns if c.startswith("sites_")]
df_final[site_cols] = df_final[site_cols].astype(int)

# Add coordination class column
def compute_coord_class(motif):
    if motif == "NONE":
        return "2His-0X"

    total_X = 0
    for token in str(motif).split("+"):
        token = token.strip()
        if not token:
            continue
        # token like ASPx1 or HISx3
        m = re.fullmatch(r"[A-Z]{3}x(\d+)", token)
        if m:
            total_X += int(m.group(1))
        else:
            total_X += 1

    if total_X >= 4:
        return "2His-4X"
    return f"2His-{total_X}X"

df_final["coord_class"] = df_final["motif"].apply(compute_coord_class)

# Sort by coordination class then by loosest-cutoff counts then motif
class_order = {"2His-0X": 0, "2His-1X": 1, "2His-2X": 2, "2His-3X": 3, "2His-4X": 4}
df_final["coord_sort"] = df_final["coord_class"].map(class_order).fillna(99)

# Sort by the largest cutoff column if present
sort_sites_col = site_cols[-1] if site_cols else None
sort_by = ["coord_sort", "motif"] if sort_sites_col is None else ["coord_sort", sort_sites_col, "motif"]
ascending = [True, False, True] if sort_sites_col is not None else [True, True]

df_final = df_final.sort_values(by=sort_by, ascending=ascending)

# Reorder columns
df_final = df_final[["coord_class", "motif"] + site_cols]

# 7) Write outputs
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

out_sites = PROJECT_ROOT / f"2His_coordination_sites_multicutoff_{timestamp}.csv"
out_summary = PROJECT_ROOT / f"2His_motif_counts_multicutoff_{timestamp}.csv"

df_sites.to_csv(out_sites, index=False)
df_final.to_csv(out_summary, index=False)

print("\nDone.")
print("Wrote site-level multicutoff table:", out_sites)
print("Wrote motif summary multicutoff table:", out_summary)

print("\nTop of final summary table:")
print(df_final.head(20))

In [ ]:
# Stratified downsampling for an SSN-ready set (Extended Data Figure 10) 
# - Uses latest 2His_coordination_classification_*.csv in PROJECT_ROOT
# - Collapses site-level -> protein-level by MOST-FREQUENT motif per protein
# - Includes ALL motif bins with >= MIN_BIN_SIZE proteins
# - Stratified sampling to ~TARGET_TOTAL
# Output: ssn_stratified_accessions_motif_<timestamp>.csv
#         (3 cols: accession_id, motif, total_multiplicity)
# --> This is used as the input for the EFI-SSN analysis 

from pathlib import Path
import pandas as pd
import numpy as np
import re
import datetime as dt

# User tunables
TARGET_TOTAL = 3000
MIN_BIN_SIZE = 10
MAX_PER_MOTIF = 250
SEED = 2026

# 1) Auto-locate latest classification file in PROJECT_ROOT
pattern = "2His_coordination_classification_*.csv"
candidates = sorted(PROJECT_ROOT.glob(pattern))
if not candidates:
    raise FileNotFoundError(f"No files matching '{pattern}' found in PROJECT_ROOT:\n{PROJECT_ROOT}")

def extract_timestamp(p: Path) -> str:
    m = re.search(r"(\d{8}_\d{6})", p.name)
    return m.group(1) if m else ""

timestamped = [p for p in candidates if extract_timestamp(p)]
if timestamped:
    candidates_sorted = sorted(timestamped, key=lambda p: extract_timestamp(p))
else:
    candidates_sorted = sorted(candidates, key=lambda p: p.stat().st_mtime)

CLASS_FILE = candidates_sorted[-1]
print("Using latest classification file:", CLASS_FILE.name)

# 2) Load + pick columns
df = pd.read_csv(CLASS_FILE)

# accession column
if "accession_id" in df.columns:
    acc_col = "accession_id"
elif "structure" in df.columns:
    acc_col = "structure"
else:
    raise ValueError(f"Can't find accession column in {CLASS_FILE.name}. Columns: {list(df.columns)}")

# motif column 
if "motif_mult" in df.columns:
    motif_col = "motif_mult"
elif "combo" in df.columns:
    motif_col = "combo"
elif "motif" in df.columns:
    motif_col = "motif"
else:
    raise ValueError(f"Can't find motif column in {CLASS_FILE.name}. Columns: {list(df.columns)}")

df = df[[acc_col, motif_col]].dropna().copy()
df.columns = ["accession_id", "motif"]
df["accession_id"] = df["accession_id"].astype(str).str.strip()
df["motif"] = df["motif"].astype(str).str.strip()

print(f"Site-level rows loaded: {len(df):,}")
print(f"Unique proteins in file: {df['accession_id'].nunique():,}")
print(f"Motif bins present (site-level): {df['motif'].nunique():,}")

# Helper: compute total multiplicity (includes the 2 His)
def total_multiplicity_from_motif(motif: str) -> int:
    """
    Returns total coordinating-ligand multiplicity, including the 2 His of the 2-His site.
    Examples:
      NONE -> 2
      ASP -> 3
      HIS+GLU -> 4  (2 + 1 + 1)
      HISx4 -> 6    (2 + 4)
      GLUx3+TYR -> 6 (2 + 3 + 1)
    """
    if motif is None:
        return 2
    m = str(motif).strip()
    if m == "" or m.upper() == "NONE":
        return 2

    extra = 0
    # split on '+', allow tokens like "HIS" or "HISx4"
    for token in m.split("+"):
        token = token.strip()
        if token == "":
            continue
        mm = re.fullmatch(r"([A-Z]{3})(?:x(\d+))?", token)
        if not mm:
            # if unexpected format, ignore rather than crash
            continue
        n = int(mm.group(2)) if mm.group(2) else 1
        extra += n

    return 2 + extra

# 2b) Collapse site-level -> ONE motif per protein
#     (most-frequent motif per protein; deterministic tie-break)
counts_pm = (
    df.groupby(["accession_id", "motif"])
      .size()
      .reset_index(name="n_sites")
)
counts_pm = counts_pm.sort_values(["accession_id", "n_sites", "motif"], ascending=[True, False, True])
df_protein_motifs = counts_pm.drop_duplicates("accession_id", keep="first")[["accession_id", "motif"]].copy()

# add multiplicity at protein-level
df_protein_motifs["total_multiplicity"] = df_protein_motifs["motif"].apply(total_multiplicity_from_motif)

print(f"\nProtein-level table: {len(df_protein_motifs):,} proteins")

# 3) Select bins with >= MIN_BIN_SIZE proteins
bin_counts = df_protein_motifs["motif"].value_counts()
selected_bins = bin_counts[bin_counts >= MIN_BIN_SIZE].index.tolist()

print(f"Motif bins with >= {MIN_BIN_SIZE} proteins: {len(selected_bins)}")
print("Top bins (protein-level):")
print(bin_counts.head(15))

df_filtered = df_protein_motifs[df_protein_motifs["motif"].isin(selected_bins)].copy()

# 4) Stratified sampling across bins
rng = np.random.default_rng(SEED)

n_bins = len(selected_bins)
if n_bins == 0:
    raise ValueError(f"No motif bins had >= {MIN_BIN_SIZE} proteins. Lower MIN_BIN_SIZE.")

base_quota = max(1, TARGET_TOTAL // n_bins)

bin_capacity = {b: min(int(bin_counts[b]), MAX_PER_MOTIF) for b in selected_bins}

remaining = TARGET_TOTAL
chosen_parts = []

# pass 1: base quota
for b in selected_bins:
    if remaining <= 0:
        break
    take = min(base_quota, bin_capacity[b], remaining)
    if take <= 0:
        continue
    sub = df_filtered[df_filtered["motif"] == b]
    idx = rng.choice(sub.index.to_numpy(), size=take, replace=False)
    chosen_parts.append(sub.loc[idx, ["accession_id", "motif", "total_multiplicity"]])
    remaining -= take
    bin_capacity[b] -= take

# pass 2: fill remaining
df_sample = (
    pd.concat(chosen_parts, ignore_index=True)
    if chosen_parts else
    pd.DataFrame(columns=["accession_id", "motif", "total_multiplicity"])
)
already = set(df_sample["accession_id"].tolist())

bins_with_left = [b for b in selected_bins if bin_capacity[b] > 0]

while remaining > 0 and bins_with_left:
    progressed = False
    for b in list(bins_with_left):
        if remaining <= 0:
            break
        if bin_capacity[b] <= 0:
            bins_with_left.remove(b)
            continue

        sub = df_filtered[df_filtered["motif"] == b]
        sub = sub[~sub["accession_id"].isin(already)]
        if len(sub) == 0:
            bin_capacity[b] = 0
            bins_with_left.remove(b)
            continue

        take = min(25, bin_capacity[b], remaining, len(sub))
        idx = rng.choice(sub.index.to_numpy(), size=take, replace=False)
        part = sub.loc[idx, ["accession_id", "motif", "total_multiplicity"]]

        df_sample = pd.concat([df_sample, part], ignore_index=True)
        already |= set(part["accession_id"].tolist())

        remaining -= take
        bin_capacity[b] -= take
        progressed = True

        if bin_capacity[b] <= 0:
            bins_with_left.remove(b)

    if not progressed:
        break

df_sample = df_sample.drop_duplicates("accession_id")

# safety cap
if len(df_sample) > TARGET_TOTAL:
    df_sample = df_sample.sample(n=TARGET_TOTAL, random_state=SEED).copy()

# 5) Summary + write output
print("\n=== Sampling summary ===")
print(f"Target total: {TARGET_TOTAL}")
print(f"Sampled total: {len(df_sample)}")
print(f"Bins represented: {df_sample['motif'].nunique()} / {len(selected_bins)}")
print("\nSampled counts by motif (top 25):")
print(df_sample["motif"].value_counts().head(25))

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = PROJECT_ROOT / f"ssn_stratified_accessions_motif_{timestamp}.csv"
df_sample[["accession_id", "motif", "total_multiplicity"]].to_csv(out_csv, index=False)

print("\nWrote SSN input (3 columns):")
print(out_csv)